In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import shelve
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import *
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
)
from pt_to_api import disjoint_ae, disjoint_ae_learned_sig
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
import numpy as np
from torch import nn
from torch import optim
import warnings
from dataclasses import dataclass
from typing import Any
import math
import gc
import pandas as pd
from pt_to_api import benchmark as B
import pt_to_api.benchmark.train_x as TX
import ast

MODE = "light"
SHELVE_CACHE_ROOT = Path.cwd() / "pure-case-v3"
SHELVE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
def weights_loss_cycled_batched(alpha, sigma_0, W_batch):
    """
    W_batch: [batch, C, K]
    Applies weights_loss_cycled logic across batch dim, returns mean over batch.
    """
    B, C, K = W_batch.shape
    W_sq = W_batch ** 2  # (B, C, K)
    
    idx = (torch.arange(K).unsqueeze(0) - torch.arange(K).unsqueeze(1)) % K  # (K, K)
    W_sq_shifted = W_sq[:, :, idx]  # (B, C, K, K)
    
    cumsum = torch.cumsum(W_sq_shifted, dim=3)              # (B, C, K, K)
    phi = alpha * torch.roll(cumsum, 1, dims=3) + 1         # (B, C, K, K)
    phi[:, :, :, 0] = 1
    
    comp1 = (W_sq_shifted * phi / (sigma_0 * sigma_0)).sum(dim=3)  # (B, C, K)
    comp2 = (-torch.log(phi)).sum(dim=3)                            # (B, C, K)
    
    # mean over K (shifts), then C (channels), then B (batch)
    return comp1.mean(), comp2.mean()


def weights_loss_cycled_batched_loop(alpha, sigma_0, W_batch):
    """
    Manual loop version for testing against batched implementation.
    W_batch: [batch, C, K]
    """
    comp1_list = []
    comp2_list = []
    for i in range(W_batch.shape[0]):
        c1, c2 = B.weights_loss_cycled(alpha, sigma_0, W_batch[i])
        comp1_list.append(c1)
        comp2_list.append(c2)
    return torch.stack(comp1_list).mean(), torch.stack(comp2_list).mean()

In [ ]:
# import time
# from torch import nn

# class Autoencoder(nn.Module):
#     def __init__(self, input_dim, n_components):
#         super().__init__()
#         self.encoder = nn.Linear(input_dim, n_components, bias=True)
#         self.decoder = nn.Linear(n_components, input_dim, bias=False)

#     def forward(self, x):
#         codes = self.encoder(x)
#         latent = codes.unsqueeze(-1) * self.decoder.weight.T.unsqueeze(0)
#         recon = latent.sum(dim=1)
#         latent_perm = latent.permute(0, 2, 1)  # [batch, dims, n_components]
#         return recon, codes, latent_perm


# def train(
#     X,
#     n_components,
#     lr=1e-3,
#     epochs=2000,
#     batch_size=64,
#     verbose=True,
#     init_strategy: B.InitStrategy=B.StandardInitStrategy(),
#     initialised_model=None,
#     use_ln_term=True,
#     sigma_s_rel_to_0="equal",
#     device="mps",
#     baseline_epochs=1000,
#     sigma_eps_override=None,
# ) -> B.SingleRun:
#     """
#     X: numpy array (n_samples, input_dim)
#     n_components: number of dictionary atoms
#     alpha: lorentzian sigma shrinker parameter
#     """
#     print("using hyperparameters")
#     print("\tinit_strategy", init_strategy)
#     print("\tuse_ln", use_ln_term)
#     print("\tsigma_s_rel_to_0", sigma_s_rel_to_0)
    
#     X_t = torch.tensor(X, dtype=torch.float32)
#     n_samples, input_dim = X_t.shape
#     X_t = X_t.to(device)

#     baseline_run = train_baseline(X, n_components, lr, baseline_epochs, batch_size, verbose)
#     sigma_eps = np.sqrt(baseline_run.loss)
#     tol = X.std() / 10_000
#     if sigma_eps < tol:
#         print(f"WARM: sigma_eps={sigma_eps} is less than tolerance={tol}, this can have undesired behavior")
#         sigma_eps = tol
#     if sigma_eps_override is not None:
#         sigma_eps = sigma_eps_override

#     print("baseline MSE", baseline_run.loss)

#     if initialised_model is None:
#         model = Autoencoder(input_dim, n_components)
#     else:
#         model = initialised_model

#     model = model.to(device)


#     scaled_hyperparameters = B.get_hyperparameters_and_init(
#         model, X, n_components, input_dim, sigma_eps, init_strategy, sigma_s_rel_to_0, device
#     )

#     p = scaled_hyperparameters
#     sigma_x = X.std()
#     sigma_eps, sigma_0, sigma_s, _, alpha = (
#         p["sigma_eps"],
#         p["sigma_0"],
#         p["sigma_s"],
#         p["sigma_enc"],
#         p["alpha"],
#     )
#     print("using hyperparameters:", p)

#     optimizer = optim.Adam(model.parameters(), lr=lr)
#     if isinstance(init_strategy, B.WarmupInitStrategy):
#         B.warmup_with_l2(
#             X_t,
#             model,
#             init_strategy.warmup_epochs,
#             optimizer,
#             batch_size,
#             sigma_eps,
#             sigma_s,
#             sigma_0,
#             verbose=verbose,
#             device=device,
#         )


#     last_print_time = time.time()

#     for epoch in range(epochs):
#         # shuffle
#         idx = torch.randperm(n_samples, device=device)
#         permuted_X_t = X_t[idx]

#         for i in range(0, n_samples, batch_size):
#             batch = permuted_X_t[i : i + batch_size]
#             recon, codes, latent_perm = model(batch)

#             _recon_loss = B.recon_loss(batch, recon, sigma_eps)
#             # _codes_loss = B.codes_loss(codes, sigma_s)
#             # comp1, comp2 = weights_loss_cycled(alpha, sigma_0, model.decoder.weight)
#             comp1, comp2 = weights_loss_batched(alpha, sigma_x, latent_perm)

#             if use_ln_term:
#                 weight_loss = comp1 + comp2
#             else:
#                 weight_loss = comp1

#             loss = _recon_loss + weight_loss

#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()

#         if verbose and epoch % 50 == 0:
#             print(
#                 f"epoch {epoch:4d} | recon_loss {_recon_loss:.4f} weight_loss {weight_loss:.4f} duration={time.time() - last_print_time}"
#             )
#             last_print_time = time.time()

#     with torch.no_grad():
#         recon, codes, _ = model(X_t)

#     return B.SingleRun(
#         model.to("cpu"),
#         codes.to("cpu").numpy(),
#         model.decoder.weight.T.detach().to("cpu").numpy(),
#         recon.to("cpu").numpy(),
#         ((X_t - recon) ** 2).to("cpu").mean(),
#         scaled_hyperparameters,
#     )


# def train_baseline(
#     X,
#     n_components,
#     lr=1e-3,
#     epochs=2000,
#     batch_size=256,
#     verbose=True,
# ):
#     """Train the autoencoder with just reconstruction loss, to find an arbitrary linear model which fits the data

#     The main training code requires sigma_eps
#     the standard deviation of expected gaussian noise
#     when the curve is fitted using Y=WX
#     We can generally do a simple sweep of hyperparams
#     or use simple heuristics
#     If the data is linearly "fittable",
#     then we get a good starting point
#     using this function.
#     """
#     print(f"training baseline model, epochs={epochs}")
#     X_t = torch.tensor(X, dtype=torch.float32)
#     n_samples, input_dim = X_t.shape

#     model = Autoencoder(input_dim, n_components)

#     optimizer = optim.Adam(model.parameters(), lr=lr)
#     for epoch in range(epochs):
#         idx = torch.randperm(n_samples)
#         permuted_X_t = X_t[idx]
#         for i in range(0, n_samples, batch_size):
#             batch = permuted_X_t[i : i + batch_size]
#             recon, codes, _ = model(batch)
#             _recon_loss = B.recon_loss(batch, recon, 1)
#             loss = _recon_loss

#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()
#         if verbose and epoch % 200 == 0:
#             print(f"finetune epoch {epoch:4d} | recon_loss {_recon_loss:.4f}")

#     with torch.no_grad():
#         recon, codes, _ = model(X_t)

#     return B.SingleRun(
#         model,
#         codes.numpy(),
#         model.decoder.weight.T.detach().numpy(),
#         recon.numpy(),
#         ((X_t - recon) ** 2).mean().item(),
#         {},
#     )


# def weights_loss_batched(alpha, sigma_0, W_batch):
#     """Batched version without cyclic shifts. W_batch: [B, C, K]"""
#     W_sq = W_batch ** 2  # (B, C, K)
#     cumsum = torch.cumsum(W_sq, dim=2)  # (B, C, K)
#     phi = alpha * torch.roll(cumsum, 1, dims=2) + 1  # (B, C, K)
#     phi[:, :, 0] = 1
#     comp1 = (W_sq * phi / (sigma_0 * sigma_0)).sum(dim=2).mean()
#     comp2 = (-torch.log(phi)).sum(dim=2).mean()
#     return comp1, comp2

# def weights_loss_batched_loop(alpha, sigma_0, W_batch):
#     """Manual loop version for testing against batched implementation."""
#     comp1_list = []
#     comp2_list = []
#     for i in range(W_batch.shape[0]):
#         c1, c2 = B.weights_loss(alpha, sigma_0, W_batch[i])
#         comp1_list.append(c1)
#         comp2_list.append(c2)
#     return torch.stack(comp1_list).mean(), torch.stack(comp2_list).mean()

In [ ]:
import matplotlib.lines as mlines


def plot_reconstruction_quality(df, key):
    """
    df must contain: dims, atom_ratio, n_sample_ratio, term3, and the column specified by key
    """
    dims_vals  = sorted(df["dims"].unique())
    atom_vals  = sorted(df["atom_ratio"].unique())
    term3_vals = sorted(df["term3"].unique())

    colors  = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63", "#9C27B0"]
    markers = ["o", "s", "^", "D", "v"]

    color_map  = {a: colors[i] for i, a in enumerate(atom_vals)}
    marker_map = {a: markers[i] for i, a in enumerate(atom_vals)}

    n_rows = len(dims_vals)
    n_cols = len(term3_vals)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        sharex=False,
        sharey="row",
        constrained_layout=True
    )
    if n_cols == 1 and n_rows == 1:
        axes = np.array([axes])
    if n_cols == 1:
        axes = axes.reshape(-1, 1)
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for row, dims in enumerate(dims_vals):
        for col, term3 in enumerate(term3_vals):
            sub = df[(df["dims"] == dims) & (df["term3"] == term3)]
            ax  = axes[row, col]

            for atom in atom_vals:
                d = sub[sub["atom_ratio"] == atom].sort_values("n_sample_ratio")
                ax.plot(
                    d["n_sample_ratio"], d[key],
                    marker=marker_map[atom], color=color_map[atom],
                    linewidth=2, markersize=7,
                    label=f"atom_ratio={atom}"
                )

            if row == 0:
                ax.set_title(f"term3 = {term3}", fontsize=12, fontweight="bold")
            if col == 0:
                ax.set_ylabel(f"dims={dims}\n{key}", fontsize=10)

            ax.set_xlabel("n_sample_ratio", fontsize=10)
            ax.grid(True, which="both", linestyle="--", alpha=0.4)
            ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
            ax.set_xticks(sorted(df["n_sample_ratio"].unique()))

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(atom_vals), fontsize=10,
               bbox_to_anchor=(0.5, -0.02), frameon=True)

    fig.suptitle(f"{key} vs. sample size\nby dims and term3", fontsize=14, fontweight="bold")

    plt.savefig(f"{key}_reconstruction_quality.png", dpi=150, bbox_inches="tight")
    plt.show()

def plot_mse_vs_meansim_multi(dfs, titles, suptitle, clip_quantile=None):
    n = len(dfs)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), constrained_layout=True)
    fig.suptitle(suptitle, fontweight="bold")

    if n == 1:
        axes = [axes]

    for ax, df, title in zip(axes, dfs, titles):
        ax.scatter(df["mse"], df["mean_sim"], s=60, alpha=0.7, color="#2196F3")
        ax.set_xlim(left=0, right=df["mse"].quantile(0.95))
        ax.set_xlabel("MSE ↓", fontsize=11)
        ax.set_ylabel("mean_sim ↑", fontsize=11)
        ax.set_title(title, fontsize=12)
        if clip_quantile:
            ax.set_xlim(left=0, right=df["mse"].quantile(clip_quantile))
        ax.grid(True, linestyle="--", alpha=0.4)

    plt.savefig("mse_vs_meansim_multi.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:

def get_device(dim):
    if dim < 100:
        return "cpu"
    else:
        return "mps"



def get_metrics_df(metrics):
    mets = []
    for k, v in metrics.items():
        k = ast.literal_eval(k)
        m = v.copy()
        m["dims"] = k[0]
        m["atom_ratio"] = k[1]
        m["n_sample_ratio"] = k[2]
        m["term3"] = k[3]
        mets.append(m)
    
    df = pd.DataFrame(mets)
    df = df.drop(columns=["vec_sim"])
    
    return df

In [ ]:
import numpy as np

def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]

def generate_synthetic_patches(
    patch_dim=72,
    n_components=10,
    k=3,
    n_samples=1000,
    noise_std=0.01,
    seed=42,
    sigma_x=1,
    leak=0.0,
):
    rng = np.random.RandomState(seed)
    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))

    # add leakage into non-owned dims
    if leak > 0:
        for i, dims in enumerate(dim_partition):
            other_dims = [d for d in range(patch_dim) if d not in dims]
            W_true[i, other_dims] += leak * rng.randn(len(other_dims))

    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses at most k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        k_i = rng.randint(1, k + 1)
        idx = rng.choice(n_components, k_i, replace=False)
        codes_true[i, idx] = rng.randn(k_i)

    X = codes_true @ W_true
    scale = sigma_x / X.std()
    X *= scale
    W_true *= scale

    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition

In [ ]:
import numpy as np


def make_dim_partition(patch_dim, n_groups, seed=42):
    """Partition patch dimensions into n_groups disjoint sets."""
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_groups]) for i in range(n_groups)]


def generate_synthetic_patches(
    patch_dim=72,
    n_groups=5,
    min_atoms_per_group=2,
    max_atoms_per_group=5,
    dim_ratio=0.7,          # max fraction of group dims an atom can use
    k=2,                    # max number of active groups per sample
    n_samples=1000,
    noise_std=0.01,
    sigma_x=1.0,
    leak=0.0,
    seed=42,
):
    """
    Generate synthetic dataset where:
    - Dimensions are partitioned into disjoint groups
    - Each group has multiple atoms, each using a random subset of group dims (up to dim_ratio)
    - Within a sample, at most one atom per group is active (groups can be silent)
    - At most k groups are active per sample
    - Codes are 1 per atom (not per group)

    Returns
    -------
    X           : (n_samples, patch_dim)
    W_true      : (n_atoms_total, patch_dim)
    codes_true  : (n_samples, n_atoms_total)
    group_info  : list of dicts with keys 'group_dims', 'atom_indices', 'atom_dims'
    """
    rng = np.random.RandomState(seed)

    # 1. Partition dimensions into groups
    group_dims = make_dim_partition(patch_dim, n_groups, seed)

    # 2. Build atoms
    group_info = []
    all_atoms = []       # list of 1D arrays of length patch_dim
    atom_to_group = []   # which group each atom belongs to

    for g, dims in enumerate(group_dims):
        n_atoms = rng.randint(min_atoms_per_group, max_atoms_per_group + 1)
        atom_indices = []
        atom_dims_list = []

        for _ in range(n_atoms):
            # Pick a random subset of this group's dims, up to dim_ratio
            max_dims = max(1, int(np.floor(dim_ratio * len(dims))))
            n_dims = rng.randint(1, max_dims + 1)
            chosen_dims = rng.choice(dims, size=n_dims, replace=False)

            atom = np.zeros(patch_dim)
            atom[chosen_dims] = rng.randn(n_dims)
            atom /= np.linalg.norm(atom) + 1e-8

            atom_idx = len(all_atoms)
            all_atoms.append(atom)
            atom_to_group.append(g)
            atom_indices.append(atom_idx)
            atom_dims_list.append(list(chosen_dims))

        group_info.append({
            'group_dims': dims,
            'atom_indices': atom_indices,
            'atom_dims': atom_dims_list,
        })

    W_true = np.stack(all_atoms, axis=0)  # (n_atoms_total, patch_dim)
    n_atoms_total = len(all_atoms)

    # Optional leakage into non-owned dims
    if leak > 0:
        for i in range(n_atoms_total):
            g = atom_to_group[i]
            owned = set(group_info[g]['atom_dims'][
                group_info[g]['atom_indices'].index(i)
            ])
            other_dims = [d for d in range(patch_dim) if d not in owned]
            W_true[i, other_dims] += leak * rng.randn(len(other_dims))
        # re-normalise after leakage
        norms = np.linalg.norm(W_true, axis=1, keepdims=True)
        W_true /= norms + 1e-8

    # 3. Build codes — at most one atom active per group, at most k groups active
    codes_true = np.zeros((n_samples, n_atoms_total))

    for i in range(n_samples):
        # Choose how many groups are active (0 to k)
        k_i = rng.randint(1, k + 1)
        active_groups = rng.choice(n_groups, size=min(k_i, n_groups), replace=False)

        for g in active_groups:
            atom_indices = group_info[g]['atom_indices']
            chosen_atom = rng.choice(atom_indices)
            codes_true[i, chosen_atom] = rng.randn()

    # 4. Compose samples
    X = codes_true @ W_true  # (n_samples, patch_dim)

    # Scale to target std
    std = X.std()
    if std > 1e-8:
        scale = sigma_x / std
        X *= scale
        W_true *= scale

    # Add noise
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, group_info, atom_to_group



In [ ]:

# ---------------------------------------------------------------------------
# Quick sanity checks
# ---------------------------------------------------------------------------
X, W, codes, group_info, atom_to_group = generate_synthetic_patches(
    patch_dim=49,
    n_groups=3,
    min_atoms_per_group=2,
    max_atoms_per_group=6,
    dim_ratio=0.7,
    k=6,
    n_samples=5000,
    noise_std=0.01,
    seed=42,
    leak=0,
)

n_atoms = W.shape[0]
print(f"patch_dim     : {X.shape[1]}")
print(f"n_samples     : {X.shape[0]}")
print(f"n_atoms_total : {n_atoms}")
print()

for g, info in enumerate(group_info):
    print(f"Group {g}: {len(info['group_dims'])} dims, "
            f"{len(info['atom_indices'])} atoms")
    for local_i, (a_idx, a_dims) in enumerate(
            zip(info['atom_indices'], info['atom_dims'])):
        print(f"  atom {a_idx}: {len(a_dims)} dims "
                f"({100*len(a_dims)/len(info['group_dims']):.0f}% of group)")

print()

# Verify mutual exclusivity within groups
violations = 0
for i in range(X.shape[0]):
    for info in group_info:
        active = [a for a in info['atom_indices'] if codes[i, a] != 0.0]
        if len(active) > 1:
            violations += 1
print(f"Mutual exclusivity violations: {violations}  (should be 0)")

# Verify group disjoint support in W
group_dim_sets = [set(info['group_dims']) for info in group_info]
for gi in range(len(group_info)):
    for gj in range(gi + 1, len(group_info)):
        overlap = group_dim_sets[gi] & group_dim_sets[gj]
        if overlap:
            print(f"WARNING: groups {gi} and {gj} share dims {overlap}")
print("Group disjoint support: OK")

# Sparsity of codes
nnz = (codes != 0).sum(axis=1)
print(f"Active atoms per sample — mean: {nnz.mean():.2f}, "
        f"min: {nnz.min()}, max: {nnz.max()}")

In [ ]:
W.shape
S([c.reshape(7,7) for c in W], (20,5), 8, mode=MODE)
plt.show()

In [ ]:
scaler = B.MeanPerDimGlobalStdScaler().fit(X)
X_scaled = scaler.transform(X)
# run = train(X_scaled, 22, 1e-2, 2000, baseline_epochs=800, batch_size=256, use_ln_term=True)

In [ ]:
from pt_to_api.benchmark import train_x as TX

In [ ]:
run = TX.train_eps_model(X_scaled, 16, 1e-2)

In [ ]:
from sklearn.decomposition import FastICA

ica_estimator = FastICA(
    n_components=22, max_iter=20000, whiten="arbitrary-variance", tol=15e-5
)
ica_estimator.fit(X_scaled)

In [ ]:
ica_estimator.mixing_
_, _, sim_vector, mean_sim = B.match_atoms(W, ica_estimator.mixing_.T)
print("mean sim", mean_sim)
B.show_closest_component_of_W_for_each_component(ica_estimator.mixing_.T, W, (7,7), (10,4), 2)

In [ ]:
run = TX.train(X_scaled, 22, 1e-2, 2000, baseline_epochs=800, batch_size=256, use_ln_term=True)

In [ ]:
# we see very promising resulst with svd, many of them have 99% similarity
metrics = B.get_metrics_from_run(run, W)
print(metrics["mean_sim"], metrics["vec_sim"])
B.show_closest_component_of_W_for_each_component(run.components, W, (7,7), (10,4), 2)

In [ ]:
run.components.shape

In [ ]:
S([w.reshape(10,10) for w in W_true], 20, len(W_true), mode=MODE)
plt.show()

In [ ]:
run = B.train(
    X_scaled, 10, 1e-2, epochs=3000, baseline_epochs=2000, device="mps", init_strategy=B.SvdInitStrategy(), use_ln_term=False
)

In [ ]:
# we see very promising resulst with svd, many of them have 99% similarity
metrics = B.get_metrics_from_run(run, W_true)
metrics["mean_sim"], metrics["vec_sim"]

In [ ]:
# we see very promising resulst with svd, many of them have 99% similarity
metrics = B.get_metrics_from_run(run, W_true)
print(metrics["mean_sim"], metrics["vec_sim"])
B.show_closest_component_of_W_for_each_component(run.components, W_true, (10,10), (10,4), 2)

In [ ]:
run = B.train(
    X_scaled, 10, 1e-2, epochs=3000, baseline_epochs=2000, device="mps", use_ln_term=False
)

In [ ]:
# we see very promising resulst with svd, many of them have 99% similarity
metrics = B.get_metrics_from_run(run, W_true)
print(metrics["vec_sim"])
metrics["mean_sim"], metrics["vec_sim"]
B.show_closest_component_of_W_for_each_component(run.components, W_true, (10,10))

In [ ]:
X, W_true, codes_true, dim_partition = generate_synthetic_patches(100, 10, 10, 1000, leak=0.3)

In [ ]:
S([w.reshape(10,10) for w in W_true], 20, len(W_true), mode=MODE)
plt.show()

In [ ]:
# we see very promising resulst with svd, many of them have 99% similarity
run = B.train(
    X_scaled, 10, 1e-2, epochs=3000, baseline_epochs=2000, device="mps", use_ln_term=False
)

metrics = B.get_metrics_from_run(run, W_true)
print(metrics["vec_sim"])
metrics["mean_sim"], metrics["vec_sim"]
B.show_closest_component_of_W_for_each_component(run.components, W_true, (10,10))

In [ ]:

# we see very promising resulst with svd, many of them have 99% similarity
run = B.train(
    X_scaled, 10, 1e-2, epochs=3000, baseline_epochs=2000, device="mps", init_strategy=B.SvdInitStrategy(), use_ln_term=False
)

metrics = B.get_metrics_from_run(run, W_true)
metrics["mean_sim"], metrics["vec_sim"]
B.show_closest_component_of_W_for_each_component(run.components, W_true, (10,10))

In [ ]:
X, W_true, codes_true, dim_partition = generate_synthetic_patches(100, 10, 10, 1000, leak=0.6)

# we see very promising resulst with svd, many of them have 99% similarity
run = B.train(
    X_scaled, 10, 1e-2, epochs=3000, baseline_epochs=2000, device="mps", init_strategy=B.SvdInitStrategy(), use_ln_term=False
)

metrics = B.get_metrics_from_run(run, W_true)
metrics["mean_sim"], metrics["vec_sim"]
B.show_closest_component_of_W_for_each_component(run.components, W_true, (10,10))